# Ablation Study — Isolating FHT vs SCNN Contributions

4 combinations, 2 scenarios (S1 Ideal, S3 Inter-Subject):

| | Raw | FHT envelope |
|---|---|---|
| **Standard CNN** | ✅ done (CNN_v1) | 🔬 test here |
| **Separable CNN** | 🔬 test here | ✅ done (SCNN_v1) |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

from config import RANDOM_SEED, N_CLASSES, get_device
from src.data_splitter import scenario_1_ideal, scenario_3_inter_subject
from src.feature_extraction import fht_envelope_batch
from src.evaluation import print_report, plot_confusion_matrix

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: mps


## Model definitions (same architectures as v1 notebooks)

In [2]:
class StandardCNN(nn.Module):
    def __init__(self, in_channels=1, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x): return self.pointwise(self.depthwise(x))


class SeparableCNN(nn.Module):
    def __init__(self, in_channels=1, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            DepthwiseSeparableConv(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            DepthwiseSeparableConv(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            DepthwiseSeparableConv(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


print("Standard CNN params:", sum(p.numel() for p in StandardCNN().parameters()))
print("Separable CNN params:", sum(p.numel() for p in SeparableCNN().parameters()))

Standard CNN params: 101831
Separable CNN params: 20625


## Training utilities

In [3]:
def make_loader(X, y, batch_size=256, shuffle=True):
    X_t = torch.from_numpy(X).float().unsqueeze(1)  # (N, 1, 8, 50)
    y_t = torch.from_numpy(y).long()
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


def train_model(model, train_loader, n_epochs=30, lr=1e-3):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(n_epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb)
            loss = criterion(out, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += xb.size(0)
        scheduler.step()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{n_epochs} — loss: {total_loss/total:.4f}, acc: {correct/total:.4f}")


@torch.no_grad()
def predict(model, X):
    model.eval()
    X_t = torch.from_numpy(X).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t), batch_size=512, shuffle=False)
    preds = []
    for (xb,) in loader:
        preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(preds)


def run_experiment(model_class, X_train, y_train, X_test, y_test, title, n_epochs=30):
    model = model_class().to(DEVICE)
    loader = make_loader(X_train, y_train)
    train_model(model, loader, n_epochs=n_epochs)
    y_pred = predict(model, X_test)
    metrics = print_report(y_test, y_pred, title=title)
    return metrics

---
## Load data

In [4]:
print("Loading S1...")
X_train_s1, y_train_s1, X_test_s1, y_test_s1, _ = scenario_1_ideal()
print(f"S1 Train: {X_train_s1.shape}, Test: {X_test_s1.shape}")

print("\nLoading S3...")
X_train_s3, y_train_s3, X_test_s3, y_test_s3, info_s3 = scenario_3_inter_subject()
print(f"S3 Train: {X_train_s3.shape}, Test: {X_test_s3.shape}")

Loading S1...


Loading windows: 100%|██████████| 251/251 [00:00<00:00, 1398.06it/s]


S1 Train: (59159, 8, 50), Test: (29281, 8, 50)

Loading S3...


Loading windows: 100%|██████████| 1385/1385 [00:01<00:00, 1270.33it/s]


S3 Train: (618926, 8, 50), Test: (168232, 8, 50)


In [5]:
# Pre-compute FHT versions
print("Computing FHT envelopes...")
X_train_s1_fht = fht_envelope_batch(X_train_s1)
X_test_s1_fht = fht_envelope_batch(X_test_s1)
X_train_s3_fht = fht_envelope_batch(X_train_s3)
X_test_s3_fht = fht_envelope_batch(X_test_s3)
print("Done.")

Computing FHT envelopes...
Done.


---
## Experiment 1: CNN + FHT (new — tests whether FHT helps standard CNN)

In [6]:
print("=" * 60)
print("CNN + FHT — Scenario 1 (Ideal)")
print("=" * 60)
m_cnn_fht_s1 = run_experiment(StandardCNN, X_train_s1_fht, y_train_s1, X_test_s1_fht, y_test_s1, "CNN+FHT — S1 Ideal")

CNN + FHT — Scenario 1 (Ideal)
Epoch   1/30 — loss: 1.0581, acc: 0.6155
Epoch  10/30 — loss: 0.5063, acc: 0.8142
Epoch  20/30 — loss: 0.3696, acc: 0.8644
Epoch  30/30 — loss: 0.3076, acc: 0.8879

  CNN+FHT — S1 Ideal
  Accuracy:  0.7676
  F1-macro:  0.7701
                    precision    recall  f1-score   support

              fist       0.87      0.90      0.88      4208
         open_hand       0.79      0.69      0.74      4179
  pinch_forefinger       0.61      0.68      0.64      4204
pinch_middlefinger       0.66      0.74      0.70      4239
               two       0.74      0.76      0.75      4175
          eversion       0.87      0.80      0.84      4064
             varus       0.90      0.80      0.85      4212

          accuracy                           0.77     29281
         macro avg       0.78      0.77      0.77     29281
      weighted avg       0.78      0.77      0.77     29281



In [7]:
print("=" * 60)
print("CNN + FHT — Scenario 3 (Inter-Subject)")
print("=" * 60)
m_cnn_fht_s3 = run_experiment(StandardCNN, X_train_s3_fht, y_train_s3, X_test_s3_fht, y_test_s3, "CNN+FHT — S3 Inter-Subject", n_epochs=30)

CNN + FHT — Scenario 3 (Inter-Subject)
Epoch   1/30 — loss: 1.2360, acc: 0.5337
Epoch  10/30 — loss: 0.8186, acc: 0.6994
Epoch  20/30 — loss: 0.7323, acc: 0.7325
Epoch  30/30 — loss: 0.6878, acc: 0.7495

  CNN+FHT — S3 Inter-Subject
  Accuracy:  0.4991
  F1-macro:  0.4956
                    precision    recall  f1-score   support

              fist       0.54      0.53      0.54     23888
         open_hand       0.46      0.58      0.52     23933
  pinch_forefinger       0.35      0.36      0.36     24284
pinch_middlefinger       0.43      0.30      0.35     24064
               two       0.58      0.53      0.56     23897
          eversion       0.66      0.71      0.68     24069
             varus       0.46      0.49      0.47     24097

          accuracy                           0.50    168232
         macro avg       0.50      0.50      0.50    168232
      weighted avg       0.50      0.50      0.50    168232



---
## Experiment 2: SCNN + Raw (new — tests whether SCNN works on raw signal)

In [8]:
print("=" * 60)
print("SCNN + Raw — Scenario 1 (Ideal)")
print("=" * 60)
m_scnn_raw_s1 = run_experiment(SeparableCNN, X_train_s1, y_train_s1, X_test_s1, y_test_s1, "SCNN+Raw — S1 Ideal")

SCNN + Raw — Scenario 1 (Ideal)
Epoch   1/30 — loss: 1.4323, acc: 0.4509
Epoch  10/30 — loss: 1.0693, acc: 0.6005
Epoch  20/30 — loss: 1.0022, acc: 0.6257
Epoch  30/30 — loss: 0.9689, acc: 0.6390

  SCNN+Raw — S1 Ideal
  Accuracy:  0.6291
  F1-macro:  0.6332
                    precision    recall  f1-score   support

              fist       0.71      0.59      0.64      4208
         open_hand       0.55      0.58      0.56      4179
  pinch_forefinger       0.53      0.57      0.55      4204
pinch_middlefinger       0.52      0.59      0.56      4239
               two       0.57      0.62      0.59      4175
          eversion       0.81      0.71      0.76      4064
             varus       0.80      0.75      0.77      4212

          accuracy                           0.63     29281
         macro avg       0.64      0.63      0.63     29281
      weighted avg       0.64      0.63      0.63     29281



In [9]:
print("=" * 60)
print("SCNN + Raw — Scenario 3 (Inter-Subject)")
print("=" * 60)
m_scnn_raw_s3 = run_experiment(SeparableCNN, X_train_s3, y_train_s3, X_test_s3, y_test_s3, "SCNN+Raw — S3 Inter-Subject", n_epochs=30)

SCNN + Raw — Scenario 3 (Inter-Subject)
Epoch   1/30 — loss: 1.5438, acc: 0.3957
Epoch  10/30 — loss: 1.3104, acc: 0.5048
Epoch  20/30 — loss: 1.2731, acc: 0.5211
Epoch  30/30 — loss: 1.2564, acc: 0.5280

  SCNN+Raw — S3 Inter-Subject
  Accuracy:  0.4431
  F1-macro:  0.4371
                    precision    recall  f1-score   support

              fist       0.43      0.45      0.44     23888
         open_hand       0.37      0.39      0.38     23933
  pinch_forefinger       0.36      0.26      0.30     24284
pinch_middlefinger       0.38      0.40      0.39     24064
               two       0.46      0.45      0.46     23897
          eversion       0.56      0.72      0.63     24069
             varus       0.49      0.44      0.46     24097

          accuracy                           0.44    168232
         macro avg       0.44      0.44      0.44    168232
      weighted avg       0.44      0.44      0.44    168232



---
## Ablation Results — Full 2×2 Table

In [10]:
print("\n" + "=" * 70)
print("ABLATION STUDY RESULTS")
print("=" * 70)

print("\n--- Scenario 1 (Ideal) ---")
print(f"{'':20s} {'Raw':>12s} {'FHT':>12s} {'Δ (FHT-Raw)':>12s}")
print("-" * 58)

cnn_raw_s1 = 0.7578  # from CNN_v1
cnn_fht_s1 = m_cnn_fht_s1['accuracy']
scnn_raw_s1 = m_scnn_raw_s1['accuracy']
scnn_fht_s1 = 0.7180  # from SCNN_v1

print(f"{'CNN (101K params)':<20s} {cnn_raw_s1:>11.2%} {cnn_fht_s1:>11.2%} {cnn_fht_s1-cnn_raw_s1:>+11.2%}")
print(f"{'SCNN (20K params)':<20s} {scnn_raw_s1:>11.2%} {scnn_fht_s1:>11.2%} {scnn_fht_s1-scnn_raw_s1:>+11.2%}")
print(f"{'Δ (SCNN-CNN)':<20s} {scnn_raw_s1-cnn_raw_s1:>+11.2%} {scnn_fht_s1-cnn_fht_s1:>+11.2%}")

print("\n--- Scenario 3 (Inter-Subject) ---")
print(f"{'':20s} {'Raw':>12s} {'FHT':>12s} {'Δ (FHT-Raw)':>12s}")
print("-" * 58)

cnn_raw_s3 = 0.5290  # from CNN_v1
cnn_fht_s3 = m_cnn_fht_s3['accuracy']
scnn_raw_s3 = m_scnn_raw_s3['accuracy']
scnn_fht_s3 = 0.4851  # from SCNN_v1

print(f"{'CNN (101K params)':<20s} {cnn_raw_s3:>11.2%} {cnn_fht_s3:>11.2%} {cnn_fht_s3-cnn_raw_s3:>+11.2%}")
print(f"{'SCNN (20K params)':<20s} {scnn_raw_s3:>11.2%} {scnn_fht_s3:>11.2%} {scnn_fht_s3-scnn_raw_s3:>+11.2%}")
print(f"{'Δ (SCNN-CNN)':<20s} {scnn_raw_s3-cnn_raw_s3:>+11.2%} {scnn_fht_s3-cnn_fht_s3:>+11.2%}")

print("\n" + "=" * 70)
print("INTERPRETATION GUIDE:")
print("  Row delta (FHT-Raw):  positive = FHT helps, negative = FHT hurts")
print("  Col delta (SCNN-CNN): positive = SCNN better, negative = SCNN worse")
print("  The 'best' cell is the one with highest accuracy.")
print("  The 'best trade-off' is highest accuracy per parameter.")
print("=" * 70)


ABLATION STUDY RESULTS

--- Scenario 1 (Ideal) ---
                              Raw          FHT  Δ (FHT-Raw)
----------------------------------------------------------
CNN (101K params)         75.78%      76.76%      +0.98%
SCNN (20K params)         62.91%      71.80%      +8.89%
Δ (SCNN-CNN)             -12.87%      -4.96%

--- Scenario 3 (Inter-Subject) ---
                              Raw          FHT  Δ (FHT-Raw)
----------------------------------------------------------
CNN (101K params)         52.90%      49.91%      -2.99%
SCNN (20K params)         44.31%      48.51%      +4.20%
Δ (SCNN-CNN)              -8.59%      -1.40%

INTERPRETATION GUIDE:
  Row delta (FHT-Raw):  positive = FHT helps, negative = FHT hurts
  Col delta (SCNN-CNN): positive = SCNN better, negative = SCNN worse
  The 'best' cell is the one with highest accuracy.
  The 'best trade-off' is highest accuracy per parameter.
